# PromptTemplate — 프롬프트를 재사용 가능한 틀로 만들기

Ch01에서 LLM 호출을 배웠다면, 이제 프롬프트를 체계적으로 관리하는 법을 배워요.

매번 프롬프트를 하드코딩하는 대신, 변수 슬롯이 있는 **템플릿(template)**을 만들어두면 같은 구조의 프롬프트를 여러 상황에서 재사용할 수 있어요.

## 학습 목표

- `PromptTemplate`으로 변수를 포함한 재사용 가능한 프롬프트를 만들 수 있어요
- `ChatPromptTemplate`으로 System / Human / AI 역할을 구분한 대화형 프롬프트를 구성할 수 있어요
- `partial()`과 동적 함수 변수로 프롬프트를 더 유연하게 다룰 수 있어요

## 사전 지식

- Ch01의 `ChatOpenAI` 기본 호출 방법
- LCEL 파이프라인 (`|` 연산자) 기초

---

> **PromptTemplate 동작 흐름** — 아래 다이어그램은 변수가 어떻게 최종 프롬프트로 조립되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart LR
    V1["변수 role<br/>'파이썬 전문가'"]:::input
    V2["변수 topic<br/>'데코레이터'"]:::input
    T["PromptTemplate<br/>템플릿 문자열<br/>'당신은 {role}입니다.<br/>{topic}에 대해 설명해주세요.'"]:::process
    P["완성된 프롬프트<br/>'당신은 파이썬 전문가입니다.<br/>데코레이터에 대해 설명해주세요.'"]:::output
    LLM["LLM<br/>ChatOpenAI"]:::external
    R["응답"]:::output

    V1 --> T
    V2 --> T
    T -->|"format() / invoke()"| P
    P --> LLM
    LLM --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

## 1. PromptTemplate

`PromptTemplate`은 문자열 기반의 간단한 프롬프트를 생성하는 템플릿입니다.

### 주요 특징:
- 중괄호 `{}`를 사용하여 변수 정의
- `from_template()` 메서드로 간편하게 생성
- `format()` 또는 `invoke()`로 변수에 값을 채워 최종 프롬프트 생성


In [ ]:
# 필수 라이브러리 import
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 모델 초기화
# temperature=0: 일관된 출력을 위해 설정
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 1.1 기본 사용법


In [ ]:
# 1단계: 템플릿 정의 - 중괄호 {}로 변수 지정


# 2단계: PromptTemplate 생성


# 3단계: format() 메서드로 변수에 값을 채워 최종 프롬프트 생성



### 1.2 체인과 함께 사용하기

`PromptTemplate`을 LLM과 연결하여 체인을 구성할 수 있습니다.


In [ ]:
# 1단계: 템플릿 정의


# 2단계: 체인 구성 (프롬프트 -> 모델 -> 파서)


# 3단계: 체인 실행



## 2. ChatPromptTemplate — 역할을 구분한 대화형 프롬프트

`PromptTemplate`이 단순 문자열이라면, `ChatPromptTemplate`은 **역할(role)** 단위로 메시지를 구성해요.
LLM은 각 역할을 구분해서 읽기 때문에, 동일한 내용이라도 역할 배치에 따라 응답 품질이 달라져요.

아래 구조도는 세 가지 역할이 어떻게 조합되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart TD
    SV["변수 role<br/>'파이썬 프로그래밍 강사'"]:::input
    HV["변수 question<br/>'리스트와 튜플의 차이점은?'"]:::input

    SM["SystemMessage<br/>'당신은 {role}입니다.<br/>항상 친절하고 명확하게 답변합니다.'"]:::process
    HM["HumanMessage<br/>'{question}'"]:::process

    CP["ChatPromptTemplate<br/>메시지 배열 조립"]:::process

    LLM["LLM<br/>ChatOpenAI"]:::external
    R["AIMessage<br/>응답"]:::output

    SV --> SM
    HV --> HM
    SM --> CP
    HM --> CP
    CP --> LLM
    LLM --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

### 주요 역할(role) 종류

| 역할 | 튜플 키 | 역할 설명 |
|------|---------|-----------|
| System | `"system"` | LLM의 행동 방식·페르소나 정의 |
| Human | `"human"` | 사용자 질문·요청 |
| AI | `"ai"` | LLM의 이전 응답 (Few-Shot에 활용) |

### 2.1 기본 사용법 - 시스템과 사용자 메시지


In [ ]:
# 1단계: ChatPromptTemplate 정의
#        - system: LLM의 역할과 행동 방식 정의
#        - human: 사용자의 질문


# 2단계: 체인 구성


# 3단계: 실행



### 2.2 대화 이력 포함하기 - Few-Shot 예제

AI의 이전 응답을 포함하여 대화의 맥락을 제공할 수 있습니다. 이는 Few-Shot Learning에 유용합니다.


In [ ]:
# 1단계: 대화 이력을 포함한 ChatPromptTemplate 정의
#        - system: 역할 정의
#        - human/ai: 예시 대화 (Few-Shot 예제)
#        - human: 실제 질문


# 2단계: 체인 구성


# 3단계: 실행



### 2.3 복잡한 프롬프트 - 실용 예제

실제 업무에서 사용할 수 있는 구조화된 프롬프트 예제입니다.


In [ ]:
# ---------------------------------------------------
# 복잡한 ChatPromptTemplate 활용 - 구조화된 리뷰 분석
# ---------------------------------------------------

# 시나리오: 고객 리뷰를 분석하여 요약 보고서를 작성하는 시스템

# 1단계: 복잡한 ChatPromptTemplate 정의
#        - system: 분석 전문가 역할 + 출력 형식 지정
#        - human: 제품명, 카테고리, 리뷰 텍스트를 변수로 전달


# 2단계: 체인 구성


# 3단계: 실행



## 3. 템플릿 변수 활용법 — partial()과 동적 변수

> **실무 팁:** 같은 템플릿을 여러 팀에서 쓴다면 `partial()`로 공통 변수를 미리 고정해두면 중복 코드를 없앨 수 있어요.

프롬프트 템플릿에서 변수를 효과적으로 사용하는 다양한 방법을 살펴볼게요.

### 3.1 부분 변수 채우기 - partial()

일부 변수를 미리 고정하고, 나중에 나머지 변수만 채울 수 있습니다.


In [ ]:
# ---------------------------------------------------
# partial() - 자주 쓰는 변수를 미리 고정하여 재사용
# ---------------------------------------------------

# 시나리오: 특정 회사의 여러 부서에서 같은 템플릿을 사용하지만,
#           회사명은 항상 동일하게 유지

# 1단계: 템플릿 정의


# 2단계: 회사명을 미리 고정 (partial)
# partial(): 자주 변하지 않는 변수를 미리 채워 재사용 가능한 프롬프트 생성


# 3단계: 나머지 변수만 채워서 사용
#        - partial_prompt는 이미 company_name이 고정된 상태
#        - department와 employee_name만 바꿔서 여러 부서에 재사용



### 3.2 동적 변수 — 함수로 자동 생성하기

> **주의:** `partial(current_time=get_current_time)`처럼 함수를 **호출하지 않고** 전달해야 해요. `get_current_time()`처럼 괄호를 붙이면 값이 고정되어버려요.

변수 값을 함수로 동적으로 생성할 수 있어요. 시간·날짜·계산 값 등을 자동으로 채울 때 유용합니다.

In [ ]:
from datetime import datetime

# 시나리오: 일일 보고서를 자동으로 생성하는 시스템
#           현재 시간을 자동으로 포함

# 1단계: 현재 시간을 반환하는 함수 정의


# 2단계: 템플릿 정의 및 함수를 partial로 연결
# partial_variables에 함수를 전달하면 실행 시점에 함수가 호출됨


# 3단계: 체인 구성


# 4단계: 실행 - 실행할 때마다 현재 시간이 자동으로 채워짐



### 3.3 템플릿 조합하기

여러 템플릿을 조합하여 복잡한 프롬프트를 구성할 수 있습니다.


In [ ]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# ---------------------------------------------------
# 템플릿 조합 - 역할별로 따로 정의하고 나중에 합치기
# ---------------------------------------------------

# 시나리오: 시스템 메시지와 사용자 메시지를 별도로 관리하고 조합

# 1단계: 각 역할별로 템플릿 정의
# SystemMessagePromptTemplate: LLM의 역할과 응답 스타일 정의

# HumanMessagePromptTemplate: 사용자 질문 템플릿


# 2단계: 템플릿 조합
# from_messages()에 객체 직접 전달 가능 (튜플 문자열 대신)


# 3단계: 체인 구성


# 4단계: 실행



## 4. 실전 예제: 다국어 번역 시스템

여러 개념을 종합하여 실용적인 번역 시스템을 구축해봅니다.


In [ ]:
# ---------------------------------------------------
# 실전 예제 - 다국어 번역 시스템
# ---------------------------------------------------

# 시나리오: 비즈니스 문서를 여러 언어로 번역하는 시스템
#           문맥을 고려하고 전문 용어를 정확하게 번역

# 1단계: 번역 프롬프트 템플릿 정의
#        - industry, tone: partial()로 재사용 가능하도록 변수화
#        - source_lang, target_lang, text: 호출마다 달라지는 변수


# 2단계: 체인 구성


# 3단계: 한국어 -> 영어 번역


# 4단계: 한국어 -> 일본어 번역



## 5. 핵심 정리


In [11]:
print("💡 핵심 정리")
print("=" * 60)
print()
print("📌 PromptTemplate vs ChatPromptTemplate")
print()
print("PromptTemplate:")
print("  - 단순 텍스트 프롬프트")
print("  - 문자열 기반")
print("  - 역할 구분 없음")
print("  - 사용: 간단한 질문, 텍스트 생성")
print()
print("ChatPromptTemplate:")
print("  - 대화형 프롬프트")
print("  - 메시지 배열 기반")
print("  - system, human, ai 역할 구분")
print("  - 사용: 대화형 AI, Few-Shot 학습")
print()
print("=" * 60)
print("📌 주요 메서드")
print("=" * 60)
print("  • from_template() - 템플릿 문자열로부터 생성")
print("  • format() - 변수를 채워 문자열 반환")
print("  • invoke() - 변수를 채워 메시지 객체 반환")
print("  • partial() - 일부 변수를 미리 고정")
print()
print("=" * 60)
print("📌 언제 사용할까?")
print("=" * 60)
print()
print("✅ PromptTemplate을 사용:")
print("  - 간단한 텍스트 생성 작업")
print("  - 역할 구분이 필요 없는 경우")
print()
print("✅ ChatPromptTemplate을 사용:")
print("  - 대화형 인터페이스 구축")
print("  - 시스템/사용자 역할 명확히 구분")
print("  - Few-Shot 예제 포함")
print("  - 복잡한 프롬프트 엔지니어링")


💡 핵심 정리

📌 PromptTemplate vs ChatPromptTemplate

PromptTemplate:
  - 단순 텍스트 프롬프트
  - 문자열 기반
  - 역할 구분 없음
  - 사용: 간단한 질문, 텍스트 생성

ChatPromptTemplate:
  - 대화형 프롬프트
  - 메시지 배열 기반
  - system, human, ai 역할 구분
  - 사용: 대화형 AI, Few-Shot 학습

📌 주요 메서드
  • from_template() - 템플릿 문자열로부터 생성
  • format() - 변수를 채워 문자열 반환
  • invoke() - 변수를 채워 메시지 객체 반환
  • partial() - 일부 변수를 미리 고정

📌 언제 사용할까?

✅ PromptTemplate을 사용:
  - 간단한 텍스트 생성 작업
  - 역할 구분이 필요 없는 경우

✅ ChatPromptTemplate을 사용:
  - 대화형 인터페이스 구축
  - 시스템/사용자 역할 명확히 구분
  - Few-Shot 예제 포함
  - 복잡한 프롬프트 엔지니어링


## 6. 추가 연습 문제

직접 해보면서 학습 내용을 정리해봅시다!


### 연습 1: 이메일 자동 생성기

**과제**: 고객에게 보낼 이메일을 자동으로 생성하는 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` 사용
- 시스템 메시지: "당신은 고객 서비스 담당자입니다. 정중하고 친절하게 작성합니다."
- 변수: `customer_name`, `order_number`, `issue_type`, `resolution`
- 이메일 형식으로 출력

**힌트**: 아래 셀에 코드를 작성해보세요.


In [ ]:
# ============================================================
# TODO: ChatPromptTemplate으로 고객 이메일 자동 생성기를 만드세요
# 힌트: from_messages([("system", ...), ("human", ...)])로 구성하고
#       customer_name, order_number, issue_type, resolution 변수를 활용하세요
# 예상 결과: 고객명·주문번호·문의유형·해결방안이 채워진 이메일 텍스트
# ============================================================

# 여기에 코드를 작성하세요

# email_prompt = ChatPromptTemplate.from_messages([
#     ...
# ])

### 연습 1 풀이

In [ ]:
# 연습 1 풀이: 이메일 자동 생성기



### 연습 2: 코드 리뷰 봇

**과제**: 코드 리뷰를 수행하는 AI 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` 사용
- Few-Shot 예제 2개 포함 (좋은 코드 예시, 개선이 필요한 코드 예시)
- 변수: `programming_language`, `code`
- 리뷰 형식: 장점, 개선점, 제안사항


In [ ]:
# ============================================================
# TODO: Few-Shot 예제 2개를 포함한 코드 리뷰 봇을 만드세요
# 힌트: ChatPromptTemplate.from_messages()에 ("human", ...), ("ai", ...) 쌍으로
#       예제를 넣고, 마지막에 실제 질문 ("human", "{programming_language}... {code}...")를 추가하세요
# 예상 결과: 장점 / 개선점 / 제안사항 형식의 코드 리뷰 결과
# ============================================================

# 여기에 코드를 작성하세요

# code_review_prompt = ChatPromptTemplate.from_messages([
#     ...
# ])

### 연습 2 풀이

In [ ]:
# 연습 2 풀이: 코드 리뷰 봇



### 연습 3: 동적 시간 포함 알림 시스템

**과제**: 현재 시간을 자동으로 포함하는 알림 메시지 생성 시스템을 만드세요.

**요구사항**:
- `PromptTemplate` 사용
- `partial()`을 사용하여 현재 시간을 동적으로 추가
- 변수: `user_name`, `task`, `deadline`
- 함수로 현재 시간 생성


In [ ]:
# ============================================================
# TODO: partial()과 동적 함수 변수를 사용하여 현재 시간이 포함된 알림 시스템을 만드세요
# 힌트: get_current_time() 함수를 정의하고 prompt.partial(current_time=get_current_time)으로
#       함수를 전달하세요 (함수를 호출하지 않고 전달해야 실행 시점의 시간이 반영됩니다)
# 예상 결과: 알림 시간이 자동으로 채워진 업무 알림 메시지
# ============================================================

# 여기에 코드를 작성하세요

# def get_current_time():
#     ...
#
# notification_prompt = PromptTemplate.from_template(...)
# notification_prompt = notification_prompt.partial(...)

### 연습 3 풀이

In [ ]:
# 연습 3 풀이: 동적 시간 포함 알림 시스템



## 마무리 — 이번 노트북에서 배운 것

| 개념 | 핵심 메서드 | 언제 쓰나요? |
|------|------------|-------------|
| `PromptTemplate` | `from_template()`, `format()` | 단순 텍스트 생성, 역할 구분 불필요 시 |
| `ChatPromptTemplate` | `from_messages()`, `invoke()` | System/Human/AI 역할을 구분한 대화형 프롬프트 |
| `partial()` | `prompt.partial(key=value)` | 공통 변수를 미리 고정해 재사용 |
| 동적 함수 변수 | `partial(key=fn)` | 시간·날짜처럼 실행 시점에 계산되는 값 |

---

**다음 노트북 예고:** `02-FewShotPromptTemplate` — LLM에게 예제를 보여주며 원하는 출력 형식을 학습시키는 **퓨샷(Few-Shot) 프롬프팅**을 배워요. 단순 형식 지정보다 훨씬 세밀한 제어가 가능해집니다.